# U-Net Flood Segmentation — Training Pipeline

**Person 2 (AI & Detection) — Triage Disaster Intelligence System**

## Overview
This notebook implements a reproducible PyTorch training pipeline for the 5-channel U-Net flood segmentation model.

### Architecture
- **Model:** `UNet5Ch` (encoder-decoder with skip connections; `src/detect/unet.py`)
- **Input:** `(B, 5, H, W)` tensor — channels: pre_vv, pre_vh, post_vv, post_vh, slope
- **Output:** `(B, 1, H, W)` logit → sigmoid → flood probability

### Dataset
**Target dataset: Kuro Siwo** (Bountos et al., 2024)  
URL: https://github.com/Orion-AI-Lab/KuroSiwo  
License: CC-BY-4.0

**Dataset availability check (run Section 1 first):**  
Kuro Siwo provides pre/post Sentinel-1 GRD (VV, VH) + binary flood labels.  
Slope must be derived from a co-registered Copernicus DEM-30 product.  

**If Kuro Siwo is unavailable:** This notebook uses a reproducible synthetic dataset  
that simulates plausible Sentinel-1 backscatter physics. All results from synthetic  
training are labelled **SMOKE TEST ONLY** and must NOT be reported as benchmark metrics.

### Execution modes
- **Smoke mode** (`SMOKE_MODE = True`): 5 epochs, 32 patches — verifies entire pipeline
- **Full mode** (`SMOKE_MODE = False`): configurable epochs, full dataset, GPU if available

### Reproducibility
All random seeds are fixed to 42. Hardware: CPU or CUDA if available.

In [ ]:
# ─── Configuration ─────────────────────────────────────────────────────────
SMOKE_MODE     = True   # Set False for full training
SEED           = 42
PATCH_SIZE     = 64     # pixels (64×64 patches)
BATCH_SIZE     = 8
LEARNING_RATE  = 1e-4
MAX_EPOCHS     = 5 if SMOKE_MODE else 50
EARLY_STOP_PATIENCE = 5 if SMOKE_MODE else 10
NUM_TRAIN_PATCHES   = 32  if SMOKE_MODE else 2000
NUM_VAL_PATCHES     = 8   if SMOKE_MODE else 400
MODEL_DIR      = '../data/models'
CHECKPOINT     = f'{MODEL_DIR}/unet_flood_best.pth'

import sys, os
# Ensure project root is on path when running notebook from notebooks/
sys.path.insert(0, os.path.abspath('..'))

print(f'Smoke mode: {SMOKE_MODE}')
print(f'Max epochs: {MAX_EPOCHS}, Batch size: {BATCH_SIZE}')

In [ ]:
# ─── Imports and seed setup ─────────────────────────────────────────────────
import json
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Fix all seeds for reproducibility
torch.manual_seed(SEED)
np.random.default_rng(SEED)  # NumPy generator
np.random.seed(SEED)         # Legacy NumPy seed
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

from src.detect.unet import normalize_5channel_input, _build_unet_model
from src.detect.unet import compute_segmentation_metrics, export_to_onnx

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('training')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {DEVICE}')

## 1. Dataset Check — Kuro Siwo Availability

Check whether real Kuro Siwo data is accessible before choosing training path.

**To use real Kuro Siwo data:**
1. Download from https://github.com/Orion-AI-Lab/KuroSiwo (CC-BY-4.0 license)
2. Set `KURO_SIWO_DIR` to the extracted directory path
3. Set `USE_REAL_DATA = True`

The dataset provides:
- Pre/post Sentinel-1 GRD (VV, VH) — satisfies channels 0–3
- Binary flood labels per pixel
- **Note:** Slope (channel 4) must be derived from Copernicus DEM-30, co-registered to S1 tiles

In [ ]:
# ─── Dataset availability check ─────────────────────────────────────────────
KURO_SIWO_DIR = Path('../data/raw/kuro_siwo')
USE_REAL_DATA = KURO_SIWO_DIR.exists() and any(KURO_SIWO_DIR.iterdir())

if USE_REAL_DATA:
    print(f'[REAL DATA] Kuro Siwo found at {KURO_SIWO_DIR}')
    print('Will use real dataset for training.')
else:
    print('[SYNTHETIC] Kuro Siwo not found at', KURO_SIWO_DIR)
    print('Status: SMOKE TEST ONLY — using physics-consistent synthetic patches.')
    print('To use real data: download Kuro Siwo and set KURO_SIWO_DIR.')
    print()
    print('WARNING: Metrics from synthetic training MUST NOT be reported as')
    print('benchmark performance. They verify pipeline correctness only.')

## 2. Dataset — Synthetic Flood Patches (Kuro Siwo Stand-in)

When real Kuro Siwo data is unavailable, we generate physics-consistent synthetic patches.

**Simulation logic:**
- Pre-event: Gaussian noise around -11 dB (VV) and -17 dB (VH) — typical Sentinel-1 land
- Post-event: Flood corridor injects -22 dB (VV) and -27 dB (VH) backscatter
- Slope: uniform random in [0, 25°]
- Label: `True` where post-event VV drop exceeds a physics threshold AND slope < 8°

These physics are based on:
- Schumann et al. (2011): SAR flood detection methodology
- Sentinel-1 GRD product specs (ESA, 2021)

In [ ]:
# ─── Synthetic dataset class ─────────────────────────────────────────────────
class SyntheticFloodDataset(Dataset):
    """Physics-consistent synthetic Sentinel-1 flood patch dataset.
    
    IMPORTANT: Results from models trained on this dataset are labelled
    SMOKE TEST ONLY and must not be cited as benchmark performance.
    """

    def __init__(self, n_patches: int, patch_size: int = 64, seed: int = 42) -> None:
        self.n = n_patches
        self.ps = patch_size
        self.rng = np.random.default_rng(seed)
        self.patches = [self._make_patch() for _ in range(n_patches)]

    def _make_patch(self):
        ps = self.ps
        pre_vv = self.rng.normal(-11.0, 2.5, (ps, ps)).astype(np.float32)
        pre_vh = self.rng.normal(-17.0, 3.0, (ps, ps)).astype(np.float32)
        slope  = self.rng.uniform(0.0, 25.0, (ps, ps)).astype(np.float32)

        post_vv = pre_vv.copy()
        post_vh = pre_vh.copy()
        label   = np.zeros((ps, ps), dtype=np.float32)

        # Flood corridor: 20–50% of patch width, centred vertically ± jitter
        centre = ps // 2 + self.rng.integers(-ps // 6, ps // 6)
        width  = self.rng.integers(ps // 5, ps // 2)
        r0, r1 = max(0, centre - width // 2), min(ps, centre + width // 2)

        flood_strip = (slope < 8.0) & (np.arange(ps)[:, None] >= r0) & (np.arange(ps)[:, None] < r1)
        n_flood = int(np.count_nonzero(flood_strip))
        if n_flood > 0:
            post_vv[flood_strip] = self.rng.normal(-22.0, 1.5, n_flood).astype(np.float32)
            post_vh[flood_strip] = self.rng.normal(-27.0, 1.5, n_flood).astype(np.float32)
            label[flood_strip] = 1.0

        tensor = normalize_5channel_input(pre_vv, pre_vh, post_vv, post_vh, slope)
        return tensor, label

    def __len__(self): return self.n
    def __getitem__(self, idx):
        x, y = self.patches[idx]
        return torch.from_numpy(x), torch.from_numpy(y).unsqueeze(0)


train_ds = SyntheticFloodDataset(NUM_TRAIN_PATCHES, PATCH_SIZE, seed=SEED)
val_ds   = SyntheticFloodDataset(NUM_VAL_PATCHES,   PATCH_SIZE, seed=SEED + 1)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train patches: {len(train_ds)}, Val patches: {len(val_ds)}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

# Check flood prevalence
all_labels = np.stack([y.numpy() for _, y in train_ds])
flood_frac = all_labels.mean()
print(f'Flood pixel fraction (train): {flood_frac:.3f} ({flood_frac*100:.1f}%)')

## 3. Loss Function — BCE + Dice

Combined Binary Cross-Entropy + Dice loss for flood pixels.

**Justification:**
- Flood pixels are severely underrepresented (~5–20% of pixels)
- BCE alone tends to collapse to predicting background
- Dice loss directly optimises F1/IoU, compensating for class imbalance
- Positive class weighting further addresses imbalance

In [ ]:
# ─── Loss function ───────────────────────────────────────────────────────────
class BCEDiceLoss(nn.Module):
    """Combined BCE + Dice loss for binary flood segmentation."""

    def __init__(self, bce_weight: float = 0.5, pos_weight: float = 3.0, smooth: float = 1e-6):
        super().__init__()
        self.bce_w = bce_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(pos_weight)
        )

    def forward(self, logit: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        bce_loss = self.bce(logit, target)

        prob = torch.sigmoid(logit)
        intersection = (prob * target).sum()
        dice_loss = 1.0 - (2.0 * intersection + self.smooth) / (
            prob.sum() + target.sum() + self.smooth
        )

        return self.bce_w * bce_loss + (1.0 - self.bce_w) * dice_loss


criterion = BCEDiceLoss(bce_weight=0.5, pos_weight=3.0)
print('Loss function: BCE (w=0.5) + Dice (w=0.5) with pos_weight=3.0')

## 4. Model, Optimizer, and Scheduler

In [ ]:
# ─── Model, optimizer, scheduler ────────────────────────────────────────────
torch.manual_seed(SEED)
model = _build_unet_model(in_channels=5, features=(64, 128, 256, 512)).to(DEVICE)

# Count parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'U-Net parameters: {n_params:,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
)

print(f'Optimizer: AdamW lr={LEARNING_RATE}, weight_decay=1e-4')
print(f'Scheduler: CosineAnnealingLR T_max={MAX_EPOCHS}')

## 5. Training Loop

In [ ]:
# ─── Training loop ────────────────────────────────────────────────────────────
from src.detect.unet import compute_segmentation_metrics

history = {'train_loss': [], 'val_loss': [], 'val_iou': [], 'val_f1': []}
best_val_iou    = -1.0
patience_count  = 0
best_epoch      = 0

print(f'Starting training for {MAX_EPOCHS} epochs on {DEVICE}...')
print('=' * 60)

for epoch in range(1, MAX_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss_accum = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logit = model(xb)
        loss  = criterion(logit, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss_accum += loss.item()

    mean_train_loss = train_loss_accum / len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss_accum = 0.0
    all_pred, all_gt = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logit = model(xb)
            val_loss_accum += criterion(logit, yb).item()
            prob = torch.sigmoid(logit).cpu().numpy()
            all_pred.append(prob.ravel())
            all_gt.append(yb.cpu().numpy().ravel())

    mean_val_loss = val_loss_accum / len(val_loader)
    pred_arr = np.concatenate(all_pred)
    gt_arr   = np.concatenate(all_gt)
    metrics  = compute_segmentation_metrics(pred_arr, gt_arr > 0.5, threshold=0.5)

    scheduler.step()

    history['train_loss'].append(mean_train_loss)
    history['val_loss'].append(mean_val_loss)
    history['val_iou'].append(metrics['iou'])
    history['val_f1'].append(metrics['f1_score'])

    print(f'Epoch {epoch:3d}/{MAX_EPOCHS} | '
          f'train_loss={mean_train_loss:.4f} | '
          f'val_loss={mean_val_loss:.4f} | '
          f'val_IoU={metrics["iou"]:.4f} | '
          f'val_F1={metrics["f1_score"]:.4f}')

    # Best checkpoint
    if metrics['iou'] > best_val_iou:
        best_val_iou = metrics['iou']
        best_epoch   = epoch
        torch.save(model.state_dict(), CHECKPOINT)
        patience_count = 0
        print(f'  → New best (IoU={best_val_iou:.4f}) saved to {CHECKPOINT}')
    else:
        patience_count += 1
        if patience_count >= EARLY_STOP_PATIENCE:
            print(f'Early stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs).')
            break

print('=' * 60)
print(f'Training complete. Best epoch: {best_epoch}, Best val IoU: {best_val_iou:.4f}')

mode_label = 'SMOKE TEST ONLY' if SMOKE_MODE else 'TRAINING COMPLETE'
print(f'Status: {mode_label}')
if SMOKE_MODE:
    print('NOTE: Smoke test metrics do not represent benchmark performance.')

## 6. Save Training History and Metrics

In [ ]:
# ─── Save training history ────────────────────────────────────────────────────
import json

results = {
    'smoke_mode':     SMOKE_MODE,
    'dataset':        'synthetic (Kuro Siwo stand-in)' if not USE_REAL_DATA else 'Kuro Siwo',
    'epochs_run':     len(history['train_loss']),
    'best_epoch':     best_epoch,
    'best_val_iou':   best_val_iou,
    'best_val_f1':    float(max(history['val_f1'])) if history['val_f1'] else None,
    'seed':           SEED,
    'device':         str(DEVICE),
    'training_valid': not SMOKE_MODE,  # Full metrics only valid if not smoke mode
    'history':        history,
}

results_path = Path(MODEL_DIR) / 'training_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'Training results saved: {results_path}')
print(json.dumps({k: v for k, v in results.items() if k != 'history'}, indent=2))

## 7. ONNX Export and Verification

In [ ]:
# ─── ONNX export ─────────────────────────────────────────────────────────────
from src.detect.unet import export_to_onnx, ONNX_MODEL_FILE

onnx_path = export_to_onnx(output_path=Path(MODEL_DIR) / 'unet_flood.onnx', verify=True)

if onnx_path and onnx_path.exists():
    size_mb = onnx_path.stat().st_size / 1024 / 1024
    print(f'ONNX exported: {onnx_path} ({size_mb:.1f} MB)')
    status = 'COMPLETE (trained weights)' if Path(CHECKPOINT).exists() else 'COMPLETE (architecture only — NOT trained)'
    print(f'ONNX export status: {status}')
else:
    print('ONNX export: FAILED')

## 8. Reproducibility Metadata

In [ ]:
# ─── Reproducibility metadata ─────────────────────────────────────────────────
import platform

meta = {
    'architecture':    'UNet5Ch (src/detect/unet.py)',
    'input_channels':  5,
    'channel_order':   ['pre_vv', 'pre_vh', 'post_vv', 'post_vh', 'slope'],
    'feature_widths':  [64, 128, 256, 512],
    'dataset':         'Kuro Siwo (real)' if USE_REAL_DATA else 'Synthetic (Kuro Siwo stand-in)',
    'n_train_patches': NUM_TRAIN_PATCHES,
    'n_val_patches':   NUM_VAL_PATCHES,
    'patch_size_px':   PATCH_SIZE,
    'loss':            'BCEWithLogitsLoss (pos_weight=3.0, w=0.5) + DiceLoss (w=0.5)',
    'optimizer':       f'AdamW lr={LEARNING_RATE} wd=1e-4',
    'scheduler':       f'CosineAnnealingLR T_max={MAX_EPOCHS}',
    'early_stopping':  EARLY_STOP_PATIENCE,
    'seed':            SEED,
    'device':          str(DEVICE),
    'torch_version':   torch.__version__,
    'python_version':  platform.python_version(),
    'checkpoint_path': CHECKPOINT,
    'onnx_path':       str(Path(MODEL_DIR) / 'unet_flood.onnx'),
    'smoke_mode':      SMOKE_MODE,
    'norm_stats': {
        'vv_mean': -12.0, 'vv_std': 4.5,
        'vh_mean': -18.0, 'vh_std': 4.5,
        'slope_max_deg': 45.0
    },
}

meta_path = Path(MODEL_DIR) / 'unet_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print('Reproducibility metadata saved:', meta_path)
print(json.dumps(meta, indent=2))